# VideoMAE v2 — cataract surgery pretraining (A100)

Trains the VideoMAE v2 (ViT-B/16, 224x224, 16-frame) model on the `data/` folder of surgery videos.
Metrics are logged every step and the learning curves are plotted inline below.

**Before running:** make sure this notebook sits next to the `models/`, `data/`, and `plot_utils.py` files from the repo, and set `DATA_ROOT` in the config cell to your video folder.

## 1. Install dependencies

In [1]:
# torch is already present on the A100 image; add the video reader + plotting
!pip -q install decord matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 123.5 MB/s eta 0:00:0000:010:01


## 2. Check the GPU

In [2]:
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


## 3. Config — set your data folder here

In [3]:
# Point DATA_ROOT at the folder containing all the .mp4 videos.
# Use an absolute path if the notebook isn't in the same dir as the data.
DATA_ROOT = "data"
OUT_DIR   = "checkpoints"      # checkpoints + metrics + learning_curve.pdf go here

MODEL       = "base"           # ViT-B/16
BATCH_SIZE  = 16
EPOCHS      = 800
IMG_SIZE    = 224
NUM_FRAMES  = 16

import os
os.makedirs(OUT_DIR, exist_ok=True)
print("data root:", os.path.abspath(DATA_ROOT))
print("output dir:", os.path.abspath(OUT_DIR))

data root: /content/data
output dir: /content/checkpoints


## 4. Quick data sanity check
Confirms the videos are found and decode to the right shape before committing GPU time.

In [8]:
import os
print("cwd:", os.getcwd())
print("contents:", os.listdir("."))

cwd: /content
contents: ['.config', 'checkpoints', 'sample_data']


In [5]:
import sys; sys.path.append(".")
from data.datasets import CataractVideoDataset

ds = CataractVideoDataset(DATA_ROOT, mode="video",
                          num_frames=NUM_FRAMES, img_size=IMG_SIZE, sampling_stride=4)
print("backend:", ds.backend, "| clips found:", len(ds))
clip = ds[0]
print("sample clip shape:", tuple(clip.shape), "dtype:", clip.dtype,
      "min %.2f max %.2f" % (clip.min(), clip.max()))
assert tuple(clip.shape) == (3, NUM_FRAMES, IMG_SIZE, IMG_SIZE), "unexpected clip shape"
print("OK - ready to train")

ModuleNotFoundError: No module named 'data'

## 5. Train

This calls the tested `pretrain.py`, which logs `metrics.jsonl` / `epoch_log.csv` every step and
auto-saves `learning_curve.pdf` at each checkpoint.

**Tip:** for a first smoke test, temporarily set `EPOCHS = 2` and add `--steps-per-epoch 20` to
confirm A100 speed and checkpointing, then run the full job.

In [ ]:
# run training as a subprocess so all the tested logging/plotting runs unchanged
cmd = f'''python pretrain.py \
  --data-root "{DATA_ROOT}" --data-mode video \
  --model {MODEL} --batch-size {BATCH_SIZE} --epochs {EPOCHS} \
  --img-size {IMG_SIZE} --num-frames {NUM_FRAMES} \
  --amp --num-workers 8 \
  --out-dir "{OUT_DIR}"'''
print(cmd)
get_ipython().system(cmd)

## 6. Plot the learning curves (inline)

Reads the metadata written during training and draws the curves right here. You can re-run this cell
any time — including while training is still going in another session — to see current progress.
It also refreshes `learning_curve.pdf` in the output dir.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from plot_utils import load_step_metrics, load_epoch_metrics, _smooth, make_curves_pdf

steps  = load_step_metrics(OUT_DIR)
epochs = load_epoch_metrics(OUT_DIR)
print(f"logged {len(steps)} steps across {len(epochs)} epochs")

SMOOTH = 50
fig, axes = plt.subplots(3, 1, figsize=(8, 10))

if steps:
    gs = [r["gstep"] for r in steps]; ls = [r["loss"] for r in steps]
    axes[0].plot(gs, ls, lw=0.6, alpha=0.35, label="loss (raw)")
    axes[0].plot(gs, _smooth(ls, SMOOTH), lw=1.6, label=f"loss (smoothed, w={SMOOTH})")
    axes[0].set(xlabel="global step", ylabel="reconstruction loss (MSE)",
                title="Training loss per step"); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(gs, [r["lr"] for r in steps], lw=1.4)
    axes[1].set(xlabel="global step", ylabel="learning rate",
                title="Learning-rate schedule"); axes[1].grid(alpha=0.3)

if epochs:
    axes[2].plot([e["epoch"] for e in epochs], [e["avg_loss"] for e in epochs],
                 marker="o", ms=3, lw=1.4)
    axes[2].set(xlabel="epoch", ylabel="avg reconstruction loss",
                title="Average training loss per epoch"); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# also refresh the saved PDF
make_curves_pdf(OUT_DIR)